In [1]:
with open("../data/tarot_database_rag.txt") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print(f"{len(chunks)} chunks loaded\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:80]}...\n")

467 chunks loaded

Chunk 0: Tarot Card: The Fool
Number: 0
Arcana: Major Arcana...

Chunk 1: Fortune Telling:
Watch for new projects and new beginnings | Prepare to take som...

Chunk 2: Light Meanings:
Freeing yourself from limitation | Expressing joy and youthful v...

Chunk 3: Shadow Meanings:
Being gullible and naive | Taking unnecessary risks | Failing t...

Chunk 4: Questions to Ask:
What would I do if I felt free to take a leap? | How willing a...

Chunk 5: ---...

Chunk 6: Tarot Card: The Magician
Number: 1
Arcana: Major Arcana...

Chunk 7: Fortune Telling:
A powerful man may play a role in your day | Your current situa...

Chunk 8: Light Meanings:
Taking appropriate action | Receiving guidance from a higher pow...

Chunk 9: Shadow Meanings:
Inflating your own ego | Abusing talents | Manipulating or dece...

Chunk 10: Questions to Ask:
What am I empowered to do? | How might my abilities come into ...

Chunk 11: ---...

Chunk 12: Tarot Card: The High Priestess
Number: 2
Arcana: 

In [4]:
import chromadb

client = chromadb.PersistentClient("../chroma")

# Delete collection if it already exists (so we can re-run this cell)
try:
    client.delete_collection("tarot")
except Exception:
    pass

collection = client.create_collection(name="tarot")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Added {collection.count()} chunks to the 'sunscreen_speech' collection")

Added 467 chunks to the 'sunscreen_speech' collection


In [5]:
results = collection.query(
    query_texts=["What does the Emperor card mean?"],
    n_results=3,
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i + 1}:")
    print(doc)
    print()

Result 1:
Tarot Card: The Emperor
Number: 4
Arcana: Major Arcana

Result 2:
Tarot Card: The Empress
Number: 3
Arcana: Major Arcana

Result 3:
Fortune Telling:
This card represents a young man or woman with an airy, intellectual demeanor, likely born a Capricorn, Aquarius, or Pisces, who wants to learn something new from you or have a discussion with you



In [6]:
from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)
from agents import Agent, Runner, SQLiteSession, trace
from setup import bedrock_tool

ModuleNotFoundError: No module named 'setup'

In [ ]:
tarot_agent = Agent(
    name="Tarot predictions",
    instructions="""
    You are a machine answering user's questions with the use of Tarot cards.
    You randomly take 3 cards out of 78, and then express their meanings together.
    The resulting prediction should make sense and be relevant to the question of the user.
    If you need to randomly choose cards, use the tool:
    If you need to look up cards meanings, use the tool:
    """,
    model=MODEL,
    tools=[bedrock_tool(calorie_lookup_tool.__dict__)],
)

In [ ]:
TAROT_DECK = [

# Major Arcana
"The Fool",
"The Magician",
"The High Priestess",
"The Empress",
"The Emperor",
"The Hierophant",
"The Lovers",
"The Chariot",
"Strength",
"The Hermit",
"Wheel of Fortune",
"Justice",
"The Hanged Man",
"Death",
"Temperance",
"The Devil",
"The Tower",
"The Star",
"The Moon",
"The Sun",
"Judgement",
"The World",

# Cups
"Ace of Cups",
"Two of Cups",
"Three of Cups",
"Four of Cups",
"Five of Cups",
"Six of Cups",
"Seven of Cups",
"Eight of Cups",
"Nine of Cups",
"Ten of Cups",
"Page of Cups",
"Knight of Cups",
"Queen of Cups",
"King of Cups",

# Pentacles
"Ace of Pentacles",
"Two of Pentacles",
"Three of Pentacles",
"Four of Pentacles",
"Five of Pentacles",
"Six of Pentacles",
"Seven of Pentacles",
"Eight of Pentacles",
"Nine of Pentacles",
"Ten of Pentacles",
"Page of Pentacles",
"Knight of Pentacles",
"Queen of Pentacles",
"King of Pentacles",

# Swords
"Ace of Swords",
"Two of Swords",
"Three of Swords",
"Four of Swords",
"Five of Swords",
"Six of Swords",
"Seven of Swords",
"Eight of Swords",
"Nine of Swords",
"Ten of Swords",
"Page of Swords",
"Knight of Swords",
"Queen of Swords",
"King of Swords",

# Wands
"Ace of Wands",
"Two of Wands",
"Three of Wands",
"Four of Wands",
"Five of Wands",
"Six of Wands",
"Seven of Wands",
"Eight of Wands",
"Nine of Wands",
"Ten of Wands",
"Page of Wands",
"Knight of Wands",
"Queen of Wands",
"King of Wands"
]

def draw_tarot_cards(topic: str = "general", n_cards: int = 3) -> str:
    """
    Draw tarot cards for a user reading.
    Args:
        topic: The topic of the reading, e.g. love, career, general
        n_cards: Number of cards to draw
    Returns:
        A text description of drawn cards with orientation
    """
    chosen = random.sample(TAROT_DECK, n_cards)
    
    results = []
    for i, card in enumerate(chosen, start=1):
        orientation = random.choice(["upright", "reversed"])
        results.append(f"{i}. {card} ({orientation})")
    
    return f"Tarot reading topic: {topic}\nDrawn cards:\n" + "\n".join(results)

In [ ]:
@function_tool
def draw_tarot_cards_tool(topic: str = "general") -> str:
    """
    Tool function for a RAG database to get 3 random cards from the deck.

    Args:
        topic: The question of the user.

    Returns:
        Names of the three random cards.
    """
    return draw_tarot_cards(topic)